# SVD Applications: Latent Semantics and Compression

## Learning Objectives

1. **Apply** SVD truncation for image and data compression
2. **Explain** Latent Semantic Analysis (LSA) and concept space
3. **Compute** query folding-in for concept-based retrieval
4. **Analyze** energy decay and how to choose rank $k$

## Best Rank-k Approximation

By the Eckart-Young theorem, truncating SVD to top-$k$ components gives the best rank-$k$ approximation:

$$A_k = U_k \Sigma_k V_k^\top = \sum_{i=1}^k \sigma_i u_i v_i^\top$$

**Applications:**
| Domain | Matrix $A$ | What SVD captures |
|--------|----------|------------------|
| Images | pixel matrix | Texture patterns |
| Documents | term-document | Latent topics |
| Ratings | user-item | Latent preferences |
| Networks | adjacency | Community structure |

In [ ]:
import math, random

def mat_mult(A, B):
    m,k = len(A),len(A[0]); n=len(B[0])
    return [[sum(A[i][l]*B[l][j] for l in range(k)) for j in range(n)] for i in range(m)]

def transpose(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def mat_vec(A, v):
    return [sum(A[i][j]*v[j] for j in range(len(v))) for i in range(len(A))]

def norm(v): return math.sqrt(sum(x**2 for x in v))
def normalize(v): n=norm(v); return [x/n for x in v] if n>0 else v

def svd_top_k(A, k, n_iter=200):
    """Compute top-k SVD components via repeated power iteration with deflation."""
    m, n = len(A), len(A[0])
    At = transpose(A)
    residual = [row[:] for row in A]
    components = []
    rng = random.Random(42)
    for _ in range(k):
        v = normalize([rng.gauss(0,1) for _ in range(n)])
        for _ in range(n_iter):
            u = normalize(mat_vec(residual, v))
            v = normalize(mat_vec(transpose(residual), u))
        sigma = sum(u[i]*sum(residual[i][j]*v[j] for j in range(n)) for i in range(m))
        components.append((sigma, u, v))
        # Deflate
        for i in range(m):
            for j in range(n):
                residual[i][j] -= sigma * u[i] * v[j]
    return components

# Simulate a term-document matrix (5 terms x 6 docs)
# Docs 0-2: "AI" topic, Docs 3-5: "sports" topic
A = [
    [2, 1, 2, 0, 0, 0],  # 'neural'
    [1, 2, 1, 0, 0, 0],  # 'learning'
    [1, 1, 2, 0, 0, 0],  # 'model'
    [0, 0, 0, 2, 1, 2],  # 'team'
    [0, 0, 0, 1, 3, 1],  # 'game'
]
terms = ['neural','learning','model','team','game']
docs = [f'D{i}' for i in range(6)]

comps = svd_top_k(A, k=2)
print("Top-2 SVD components:")
for i,(sigma,u,v) in enumerate(comps):
    print(f"
σ_{i+1} = {sigma:.3f}")
    print(f"  Left (terms): {dict(zip(terms,[round(x,3) for x in u]))}")
    print(f"  Right (docs): {dict(zip(docs,[round(x,3) for x in v]))}")


## Latent Semantic Analysis (LSA)

In the **concept space** (rank-$k$ SVD):

- **Documents** are represented as rows of $V_k$ (or $\Sigma_k V_k^\top$ columns)
- **Terms** are represented as rows of $U_k$ (or $U_k \Sigma_k$)
- **Similarity** between documents: cosine of their concept vectors

**Query folding-in:** to find documents similar to a query $q$ (a term vector):
$$q_k = q^\top U_k \Sigma_k^{-1}$$
Project the query into concept space and find nearest document vectors.

In [ ]:
# Project query into concept space
def dot(a, b): return sum(x*y for x,y in zip(a,b))

sigmas = [c[0] for c in comps]
Us = [c[1] for c in comps]  # k left singular vectors (each of length m=n_terms)
Vs = [c[2] for c in comps]  # k right singular vectors (each of length n=n_docs)

# Query: "neural learning" → term vector
query = [2, 1, 0, 0, 0]  # weights for [neural, learning, model, team, game]

# Project query: q_k[i] = (query . u_i) / sigma_i
query_k = [dot(query, Us[i]) / sigmas[i] for i in range(2)]
print(f"Query 'neural learning' in concept space: {[round(x,4) for x in query_k]}")

# Document vectors in concept space: d_k[j] = (v_i[j] for i in range(k))
def doc_vector(j):
    return [Vs[i][j] for i in range(2)]

# Cosine similarity to query
def cosine(a, b):
    dot_ab = dot(a, b)
    return dot_ab / (norm(a)*norm(b)) if norm(a)*norm(b) > 0 else 0

print("
Document similarities to query 'neural learning':")
for j, doc in enumerate(docs):
    dv = doc_vector(j)
    sim = cosine(query_k, dv)
    bar = '#'*int(abs(sim)*20)
    print(f"  {doc}: {sim:+.4f}  {bar}")

## Choosing Rank k

**Elbow method:** plot $\sigma_i^2$ vs $i$; choose $k$ at the "elbow" — where the curve bends.

**Variance explained:** choose $k$ such that $\sum_{i=1}^k \sigma_i^2 / \sum_i \sigma_i^2 \geq 0.9$.

**Reconstruction error:** plot $\|A - A_k\|_F$ vs $k$; diminishing returns after the elbow.

In practice:
- Images: $k = 10$–$100$ (90%+ compression)
- LSA: $k = 100$–$300$ (captures major topics)
- Recommendations: $k = 20$–$50$ (latent factors)